## Script to filter for Schools + Districts that offer grades 9 - 12th grade

### Some pre-reqs:
- Ensure that the script has access to the `School Characteristics` file and all other `.csv` files

In [ ]:
import pandas as pd

characteristics_df = pd.read_csv('School_Characteristics.csv')

#### Filter school characteristics csv file that have the following conditions:
- Offer schooling from 9 - 12 (high school), I believe we wanted to focus on this grade level from our last conversation with Caroline? 
    - `SCH_GRADE_G0`
    - `SCH_GRADE_G10`
    - `SCH_GRADE_G11`
    - `SCH_GRADE_G12`

The following  indicators can place more context behind the predicted seclusion/retention rates that we may find and we should take into consideration whether 
(for eg.) a school has a special education focus because it direclty impacts the seclusion/retention rate that could be reported by the school

- Indicator for the school having a juvenile justice facility `SCH_JUST_IND`
- Indicator for whether the school is labeled under an alternative school focus or status `SCH_ALTFOCUS` , `SCH_STATUS_ALT`
- Indicator for whether the school is a special education school `SCH_STATUS_SPED`
- Does the school offer high school grade levels that are "ungraded" ? `SCH_UGDETAIL_HS`
- Does the school offer virtual instruction? `SCH_VIRT_IND`

In [ ]:
# the main filtering should occur on schools that offer high school grade levels. ungraded hs column is full of -9
# so ill just filter on this for now since we can be sure they offer high school
characteristics_hs_df = characteristics_df[(characteristics_df['SCH_GRADE_G09'] == 'Yes') &
                                                 (characteristics_df['SCH_GRADE_G10'] == 'Yes') &
                                                 (characteristics_df['SCH_GRADE_G11'] == 'Yes') &
                                                 (characteristics_df['SCH_GRADE_G12'] == 'Yes')]

Some other features we would need to keep to keep track of what columns/rows we are using across all csv files would be:

- `LEA_STATE_NAME`
- `COMBOKEY`
- `SCHID`
- `LEA_NAME`
Maybe school name? not sure about that but i'll keep it for now -> SCH_NAME

In [ ]:
# keeping the columns I think are relevant
characteristics_hs_filtered = characteristics_hs_df[['LEA_STATE_NAME', 'COMBOKEY', 'SCHID','LEA_NAME' ,'SCH_GRADE_G09','SCH_GRADE_G10','SCH_GRADE_G11','SCH_GRADE_G12', 'SCH_JUST_IND', 'SCH_ALTFOCUS', 'SCH_STATUS_ALT', 'SCH_STATUS_SPED', 'SCH_UGDETAIL_HS', 'SCH_VIRT_IND' , ]]

In [ ]:
characteristics_hs_filtered.head()
# inserting a column to specify what year the data has come from
# Syntax: df.insert(location_index, column_name, value)
# replace with your year
characteristics_hs_filtered.insert(0, 'YEAR', '2023-2024')
characteristics_hs_filtered.to_csv("filtered_School_Characteristics.csv", index=False)

Filtering CSV files based on this criteria
Based on the combokey's provided by the characteristics_hs_filtered_df :

- We can now filter the other CSV files to only keep the relevant information pertaining to these specific schools ( the ones that offer high school )

- I'm going to use pandas' .query for this since it behaves similarly to SQL which makes it easier for me to think about how to structure the querying and cross comparison

In [ ]:
unfiltered_data = [ "Transfers.csv", "Suspensions.csv", "School_Support.csv",  "School_Characteristics.csv",
                   "Retention.csv", "Justice_Facilities.csv", "InternetAccess_and_Devices.csv",
                    "Harassment_or_Bullying.csv", "Expulsions.csv", "Enrollment.csv", "Dual_Enrollment.csv",
                    "Credit_Recovery.csv", "Corporal_Punishment.csv", "COVID_Directional_Indicators_and_Items.csv"]

for i in unfiltered_data:
  full_path = f"/content/unfiltered_data/{i}"

  df = pd.read_csv(full_path)
  df_filtered = df.query('COMBOKEY in @characteristics_hs_filtered.COMBOKEY')
  df_filtered.to_csv(f"/content/filtered_data/filtered_{i}", index=False)